#### From Text to Diagnosis:
#### Evaluating GPT-Based Models for Classifying Depression Severity in Online Texts

# Script 3 - DSPy OPTIMIZATION


We use the DSPy framework to optimize the prompt and few-shot. Here, the signature, module and optimizer are defined. Then we evaluate the program. We use DSPy's documentation to guide the implementation: https://colab.research.google.com/github/stanfordnlp/dspy/blob/main/examples/nli/scone/scone_with_MIPRO.ipynb#scrollTo=NVfMJ_FpBlSI

In [ ]:
import os
import pandas as pd
import json
import openai
import getpass
import dspy
import glob
import random
from dspy.evaluate import Evaluate
import cloudpickle as pickle
from dspy.teleprompt import MIPROv2
import re
from dspy.datasets import DataLoader
from sklearn.metrics import accuracy_score, mean_absolute_error, root_mean_squared_error, confusion_matrix, precision_score, recall_score, f1_score, classification_report





In [ ]:
# Setting the API key
api_key = getpass.getpass(prompt="OpenAI API key:") 
os.environ['OPENAI_API_KEY'] = api_key

openai.api_key = os.getenv('OPENAI_API_KEY')

In [ ]:
# Loading and setting the GPT-3.5-turbo model
turbo = dspy.OpenAI(model='gpt-3.5-turbo')
dspy.settings.configure(lm=turbo)

In [ ]:
# Loading the training and validation datasets
dl = DataLoader()
train_set= dl.from_csv(
    "C:/Users/45233/Downloads/data train.csv",
    fields=("text", "multi_label_number"),
    input_keys=("text",)
)

dev_set= dl.from_csv(
    "C:/Users/45233/Downloads/data-validation.csv",
    fields=("text", "multi_label_number"),
    input_keys=("text",)
)

In [ ]:
# Checking the datasets
print(train_set[0])
print(dev_set[0])
print(len(train_set))
print(len(dev_set))

### Signatures
Here we define the signature, which has a brief prompt containing the depression scale, and an input and output field. The text data is the input, and the depression label is the output.

In [ ]:
# Define the signature
class InputOutput(dspy.Signature):
    """Label the text from a scale from 0 to 3'. 0 indicates minimum depression, 1 indicates mild 
    depression, 2 indicates moderate depression and 3 indicates severe depression."""
    # Define the input and output fields, which is the text and the depression label
    text = dspy.InputField()
    label = dspy.OutputField(desc="0, 1, 2, or 3")


## Modules
Then we define the module. We use the Predictor module, which simply executes the signature.

In [ ]:
# Define the module
class DepressionChecker(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_answer = dspy.Predict(InputOutput)

    def forward(self, text):
        return self.generate_answer(text=text)

In [ ]:
# Testing the module
text = "It feels great, but I'm worried. I tried to kill myself one year and one month ago. I am prone to depression. I am worried about crashing too hard, and I have kids to care for. Has anyone else experienced anything like this?"
uncompiled = DepressionChecker()
uncompiled(text=text).label

In [ ]:
# See the inner workings of the module
turbo.inspect_history(1)

In [ ]:

# Validation logic: check that the predicted answer is correct. Returns a boolean
def validate_ans(example, pred, trace=None):
    def extract_numbers(text):
        return ''.join(re.findall(r'\d+', str(text)))
    # Extract numbers from the 'multi_label_number' field of the 'example' object
    example_num = extract_numbers(example.multi_label_number)
    # Extract numbers from the 'label' field of the 'pred' object
    pred_num = extract_numbers(pred.label)
    # Compare the extracted numbers and return True if they match, otherwise False
    return example_num == pred_num

## Optimization
We implement the Miprov2 optimizer. It uses the metric from above to maximize score by creating 20 different prompt and examples (num_candidates) and applied to the training set over 30 rounds (num_batches).

In [ ]:
# Define evaluation arguments 
eval_kwargs = dict(num_threads=2, display_progress=True, display_table=0)

# Initialize a MIPROv2 optimizer with the GPT-3.5-turbo model, the validation logic, and other arguments
teleprompter = MIPROv2(prompt_model=turbo, task_model = turbo, metric=validate_ans, num_candidates=20, init_temperature=0.7 ,verbose=True)

# Compile the program
compiled_program = teleprompter.compile(DepressionChecker(), trainset=train_set, valset=dev_set, num_batches=30, max_bootstrapped_demos=1,max_labeled_demos=20, eval_kwargs=eval_kwargs)

## Validation (Accuracy, MAE and RMSE)

In [ ]:
# Saving the program
#compiled_program.save("compiled_program_NLP.dspy")

# Loading the program
compiled_program = DepressionChecker()
compiled_program.load('compiled_program_NLP.dspy')

In [ ]:
# Evaluate the compiled program on the validation set
scores = []
predictions = []
true_labels = []
for x in dev_set:
    pred = compiled_program(**x.inputs())
    score = validate_ans(x, pred)
    scores.append(score)
    predictions.append(pred.label)
    true_labels.append(x.multi_label_number)


In [ ]:
# Create a dictionary with the required columns
data = {
    'multi_label_number': true_labels,
    'predictions': predictions,
    'scores': scores,
}

# Create the dataframe
df_dev = pd.DataFrame(data)

# Print the dataframe
print(df_dev)

In [ ]:
# Convert true labels to integers
true_labels = [int(label) for label in true_labels]
# Convert predictions to integers, ignoring any non-numeric values
predictions = [int(pred) for pred in predictions if str(pred).isdigit()]

# Calculating the evaluation metrics: accuracy, mean absolute error, root mean squared error, and confusion matrix
accuracy = accuracy_score(true_labels, predictions)
mae = mean_absolute_error(true_labels, predictions)
rmse = root_mean_squared_error(true_labels, predictions)
conf_matrix = confusion_matrix(true_labels, predictions)
precision = precision_score(true_labels, predictions, average='macro')
recall = recall_score(true_labels, predictions, average='macro')
f1= f1_score(true_labels, predictions, average='macro')

print("accuracy:\n", accuracy, "\n")
print("mean absolute error:\n", mae, "\n")
print("root mean squared error:\n", rmse, "\n")
print("confusion matrix:\n", conf_matrix, "\n")
print("precision:\n", precision, "\n")
print("recall:\n", recall, "\n")
print("f1 score:\n", f1, "\n")

class_report = classification_report(true_labels, predictions, target_names=['Class 0', 'Class 1', 'Class 2', 'Class 3']) 

print(class_report)


In [ ]:
# See the inner workings of the optimized program. Contains the optimized prompt and bootstrapped example
turbo.inspect_history(n=3)

In [ ]:
# Load the test set
test_set= dl.from_csv(
    "C:/Users/45233/Downloads/data test.csv",
    fields=("text", "multi_label_number"),
    input_keys=("text",)
)

In [ ]:
scores = []
predictions = []
true_labels = []
for x in test_set:
    pred = compiled_program(**x.inputs())
    score = validate_ans(x, pred)
    scores.append(score)
    predictions.append(pred.label)
    true_labels.append(x.multi_label_number)

In [ ]:
# Create a dictionary with the required columns
data = {
    'multi_label_number': true_labels,
    'predictions': predictions,
    'scores': scores,
}

# Create the dataframe
df_dev = pd.DataFrame(data)

# Print the dataframe
print(df_dev)

In [ ]:
# Convert true labels to integers
true_labels = [int(label) for label in true_labels]
# Convert predictions to integers, ignoring any non-numeric values
predictions = [int(pred) if str(pred).isdigit() else 0 for pred in predictions]

# Calculating the evaluation metrics: accuracy, mean absolute error, root mean squared error, and confusion matrix
accuracy = accuracy_score(true_labels, predictions)
mae = mean_absolute_error(true_labels, predictions)
rmse = root_mean_squared_error(true_labels, predictions)
conf_matrix = confusion_matrix(true_labels, predictions)
precision = precision_score(true_labels, predictions, average='macro')
recall = recall_score(true_labels, predictions, average='macro')
f1= f1_score(true_labels, predictions, average='macro')

print("accuracy:\n", accuracy, "\n")
print("mean absolute error:\n", mae, "\n")
print("root mean squared error:\n", rmse, "\n")
print("confusion matrix:\n", conf_matrix, "\n")
print("precision:\n", precision, "\n")
print("recall:\n", recall, "\n")
print("f1 score:\n", f1, "\n")

class_report = classification_report(true_labels, predictions, target_names=['Class 0', 'Class 1', 'Class 2', 'Class 3'])

print(class_report)
